In [1]:
import numpy as np
import SoapySDR
from SoapySDR import *
import time, os
from collections import deque

class RX_200k_Robust_Config:
    def __init__(self):
        self.fs = 200e3           
        self.center_freq = 2.4e9
        self.gain = 60.0          
        self.ant = "TX/RX"        
        self.save_path = "rx_synced_200k_robust.dat"
        self.sync_threshold = 0.38 # 适应有损信道的经验阈值
        self.record_duration = 75
        self.device_args = {"driver": "uhd", "type": "usrp2", "addr": "192.168.10.1"}

class RobustBarkerRecorder:
    def __init__(self, cfg):
        self.cfg = cfg
        barker = np.array([1, 1, 1, 1, 1, -1, -1, 1, 1, -1, 1, -1, 1])
        self.template = np.repeat(barker, 100).astype(np.complex64)
        
        self.device = SoapySDR.Device(self.cfg.device_args)
        self.device.setSampleRate(SOAPY_SDR_RX, 0, self.cfg.fs)
        self.device.setFrequency(SOAPY_SDR_RX, 0, self.cfg.center_freq)
        self.device.setGain(SOAPY_SDR_RX, 0, self.cfg.gain)
        self.device.setAntenna(SOAPY_SDR_RX, 0, self.cfg.ant)
        self.rx_stream = self.device.setupStream(SOAPY_SDR_RX, SOAPY_SDR_CF32, [0])
        
        self.pre_buffer = deque(maxlen=30)
        self.all_data = []
        self.is_rec = False

    def run(self):
        self.device.activateStream(self.rx_stream)
        buff = np.empty(8000, dtype=np.complex64) 
        total_samples = int(self.cfg.fs * self.cfg.record_duration)
        count = 0
        
        print("🔍 实时监听中... 目标: 峰值 > 0.1 且 相关性 > 0.38")
        try:
            while True:
                sr = self.device.readStream(self.rx_stream, [buff], len(buff))
                if sr.ret <= 0: continue
                chunk = buff[:sr.ret]
                current_peak = np.max(np.abs(chunk))
                
                if not self.is_rec:
                    corr = np.abs(np.correlate(chunk, self.template, mode='valid'))
                    pwr_norm = np.sqrt(np.mean(np.abs(chunk)**2) * np.mean(np.abs(self.template)**2)) + 1e-9
                    max_corr = np.max(corr) / (pwr_norm * 1300)
                    
                    if time.time() % 0.5 < 0.1:
                        status = "🟢 正常" if current_peak > 0.05 else "🔴 信号弱"
                        print(f"{status} | 峰值: {current_peak:.3f} | 相关性: {max_corr:.3f}")
                    
                    # 双重验证逻辑：排除噪声误触发
                    if max_corr > self.cfg.sync_threshold and current_peak > 0.1:
                        print(f"🚀 [精准捕获] 相关性: {max_corr:.3f} | 录制开始！")
                        self.is_rec = True
                        for p in self.pre_buffer: self.all_data.append(p)
                        self.all_data.append(chunk.copy())
                        count += (len(self.pre_buffer)*len(chunk) + len(chunk))
                    else:
                        self.pre_buffer.append(chunk.copy())
                else:
                    self.all_data.append(chunk.copy())
                    count += sr.ret
                    if count >= total_samples: break
            
            np.concatenate(self.all_data).tofile(self.cfg.save_path)
            print(f"✅ 文件录制成功，已保存至 {self.cfg.save_path}")
        except KeyboardInterrupt:
            pass
        finally:
            self.device.deactivateStream(self.rx_stream)
            self.device.closeStream(self.rx_stream)

if __name__ == "__main__":
    rec = RobustBarkerRecorder(RX_200k_Robust_Config())
    rec.run()

[INFO] [UHD] linux; GNU C++ version 11.2.0; Boost_107400; UHD_4.1.0.5-3
[INFO] [usrp2_impl.cpp:312] [USRP2] Opening a USRP2/N-Series device...
[INFO] [USRP2] Opening a USRP2/N-Series device...
[INFO] [usrp2_impl.cpp:351] [USRP2] Current recv frame size: 1472 bytes
[INFO] [USRP2] Current recv frame size: 1472 bytes
[INFO] [usrp2_impl.cpp:353] [USRP2] Current send frame size: 1472 bytes
[INFO] [USRP2] Current send frame size: 1472 bytes


🔍 实时监听中... 目标: 峰值 > 0.1 且 相关性 > 0.38
🔴 信号弱 | 峰值: 0.001 | 相关性: 0.301
🔴 信号弱 | 峰值: 0.001 | 相关性: 0.304
🔴 信号弱 | 峰值: 0.001 | 相关性: 0.327
🔴 信号弱 | 峰值: 0.001 | 相关性: 0.294
🔴 信号弱 | 峰值: 0.001 | 相关性: 0.321
🔴 信号弱 | 峰值: 0.001 | 相关性: 0.291
🔴 信号弱 | 峰值: 0.001 | 相关性: 0.327
🔴 信号弱 | 峰值: 0.001 | 相关性: 0.310
🔴 信号弱 | 峰值: 0.001 | 相关性: 0.325
🔴 信号弱 | 峰值: 0.001 | 相关性: 0.288
🔴 信号弱 | 峰值: 0.001 | 相关性: 0.313
🔴 信号弱 | 峰值: 0.001 | 相关性: 0.300
🔴 信号弱 | 峰值: 0.001 | 相关性: 0.314
🔴 信号弱 | 峰值: 0.001 | 相关性: 0.314
🔴 信号弱 | 峰值: 0.001 | 相关性: 0.307
🔴 信号弱 | 峰值: 0.001 | 相关性: 0.306
🔴 信号弱 | 峰值: 0.001 | 相关性: 0.300
🔴 信号弱 | 峰值: 0.001 | 相关性: 0.309
🔴 信号弱 | 峰值: 0.001 | 相关性: 0.306
🔴 信号弱 | 峰值: 0.001 | 相关性: 0.316
🔴 信号弱 | 峰值: 0.001 | 相关性: 0.295
🔴 信号弱 | 峰值: 0.001 | 相关性: 0.315
🔴 信号弱 | 峰值: 0.001 | 相关性: 0.300
🔴 信号弱 | 峰值: 0.001 | 相关性: 0.339
🔴 信号弱 | 峰值: 0.001 | 相关性: 0.325
🔴 信号弱 | 峰值: 0.001 | 相关性: 0.301
🔴 信号弱 | 峰值: 0.001 | 相关性: 0.320
🔴 信号弱 | 峰值: 0.001 | 相关性: 0.325
🔴 信号弱 | 峰值: 0.001 | 相关性: 0.309
🔴 信号弱 | 峰值: 0.001 | 相关性: 0.297
🔴 信号弱 | 峰值: 0.001 | 相关性: 0.302
🔴 

[ERROR] [usrp2_iface.cpp:278] [USRP2] Control packet attempt 0, sequence number 7289:
RuntimeError: no control response, possible packet loss
[ERROR] [USRP2] Control packet attempt 0, sequence number 7289:
RuntimeError: no control response, possible packet loss
[ERROR] [usrp2_iface.cpp:278] [USRP2] Control packet attempt 1, sequence number 7290:
RuntimeError: no control response, possible packet loss
[ERROR] [USRP2] Control packet attempt 1, sequence number 7290:
RuntimeError: no control response, possible packet loss
[ERROR] [usrp2_iface.cpp:278] [USRP2] Control packet attempt 2, sequence number 7291:
RuntimeError: no control response, possible packet loss
[ERROR] [USRP2] Control packet attempt 2, sequence number 7291:
RuntimeError: no control response, possible packet loss
[ERROR] [tasks.cpp:59] [UHD] An unexpected exception was caught in a task loop.The task loop will now exit, things may not work.RuntimeError: link dead: timeout waiting for control packet ACK
[ERROR] [UHD] An unexp

In [ ]:
import numpy as np
import SoapySDR
from SoapySDR import *
import time, os

class UltraSafe_Recorder_Config:
    def __init__(self):
        self.fs = 200e3           
        self.center_freq = 2.4e9  
        self.gain = 50.0          # 降低增益以防溢出
        self.save_path = "rx_synced_200k_final_clean.dat"
        self.sync_threshold = 0.65 # 提高阈值，防止幽灵触发
        self.record_duration = 75  
        self.device_args = {
            "driver": "uhd", 
            "type": "usrp2", 
            "addr": "192.168.10.1",
            "recv_buff_size": "50000000" # 强制 50MB 缓冲
        }

class AcademicStableRecorder:
    def __init__(self, cfg):
        self.cfg = cfg
        # 预生成巴克码模板
        barker = np.array([1, 1, 1, 1, 1, -1, -1, 1, 1, -1, 1, -1, 1])
        self.template = np.repeat(barker, 100).astype(np.complex64)
        
        # 初始化设备
        self.device = SoapySDR.Device(self.cfg.device_args)
        self.device.setSampleRate(SOAPY_SDR_RX, 0, self.cfg.fs)
        self.device.setFrequency(SOAPY_SDR_RX, 0, self.cfg.center_freq)
        self.device.setGain(SOAPY_SDR_RX, 0, self.cfg.gain)
        
        # ⚡ 核心改进：静态内存预分配
        self.total_samples = int(self.cfg.fs * self.cfg.record_duration)
        self.storage = np.zeros(self.total_samples, dtype=np.complex64)
        
        # 配置接收流
        st_args = {"buffering": "True"}
        self.rx_stream = self.device.setupStream(SOAPY_SDR_RX, SOAPY_SDR_CF32, [0], st_args)

    def run(self):
        self.device.activateStream(self.rx_stream)
        buff = np.empty(25000, dtype=np.complex64) # 增大读取块
        count = 0
        is_rec = False
        last_log = time.time()
        
        print(f"📡 监听中... 请启动发射机。")
        
        try:
            while True:
                sr = self.device.readStream(self.rx_stream, [buff], len(buff), timeoutUs=1000000)
                if sr.ret <= 0: continue
                
                chunk = buff[:sr.ret]
                
                if not is_rec:
                    # 降低计算频率：每 3 块计算一次相关性
                    corr = np.abs(np.correlate(chunk, self.template, mode='valid'))
                    pwr = np.sqrt(np.mean(np.abs(chunk)**2) * np.mean(np.abs(self.template)**2)) + 1e-9
                    max_corr = np.max(corr) / (pwr * 1300)
                    
                    if max_corr > self.cfg.sync_threshold and np.max(np.abs(chunk)) > 0.15:
                        print(f"🚀 [触发成功] 相关性: {max_corr:.3f} | 开始高速写入...")
                        is_rec = True
                
                if is_rec:
                    # 直接内存切片赋值，这是 Python 最快的写入方式
                    num = sr.ret
                    if count + num <= self.total_samples:
                        self.storage[count:count+num] = chunk
                        count += num
                    
                    if time.time() - last_log > 1.0:
                        print(f"💾 进度: {(count/self.total_samples)*100:.1f}% | 峰值: {np.max(np.abs(chunk)):.3f}")
                        last_log = time.time()
                    
                    if count >= self.total_samples: break
            
            # ✅ 先断开硬件，彻底杜绝 Control Packet 报错
            print("✅ 采集完成，断开硬件链路...")
            self.device.deactivateStream(self.rx_stream)
            self.device.closeStream(self.rx_stream)
            
            print(f"📦 正在持久化至磁盘...")
            self.storage.tofile(self.cfg.save_path)
            print("🏁 任务结题。")
            
        except Exception as e:
            print(f"❌ 链路崩溃: {e}")
        finally:
            try: self.device.closeStream(self.rx_stream)
            except: pass

if __name__ == "__main__":
    conf = UltraSafe_Recorder_Config()
    recorder = AcademicStableRecorder(conf)
    recorder.run()

[INFO] [UHD] linux; GNU C++ version 11.2.0; Boost_107400; UHD_4.1.0.5-3
[INFO] [usrp2_impl.cpp:312] [USRP2] Opening a USRP2/N-Series device...
[INFO] [USRP2] Opening a USRP2/N-Series device...
[INFO] [usrp2_impl.cpp:351] [USRP2] Current recv frame size: 1472 bytes
[INFO] [USRP2] Current recv frame size: 1472 bytes
[INFO] [usrp2_impl.cpp:353] [USRP2] Current send frame size: 1472 bytes
[INFO] [USRP2] Current send frame size: 1472 bytes
[INFO] [usrp2_impl.cpp:546] [USRP2] Detecting internal GPSDO.... 
[INFO] [USRP2] Detecting internal GPSDO.... 
[INFO] [gps_ctrl.cpp:243] [GPS] No GPSDO found
[INFO] [GPS] No GPSDO found


🔍 [零中断模式] 启动监听 | 采样率: 200kHz
等待中... [峰值: 0.001 | 相关性: 0.279]
等待中... [峰值: 0.001 | 相关性: 0.250]
等待中... [峰值: 0.001 | 相关性: 0.230]
等待中... [峰值: 0.001 | 相关性: 0.243]
等待中... [峰值: 0.001 | 相关性: 0.229]
等待中... [峰值: 0.001 | 相关性: 0.236]
等待中... [峰值: 0.001 | 相关性: 0.230]
等待中... [峰值: 0.001 | 相关性: 0.234]
等待中... [峰值: 0.001 | 相关性: 0.261]
等待中... [峰值: 0.001 | 相关性: 0.258]
等待中... [峰值: 0.001 | 相关性: 0.253]
等待中... [峰值: 0.001 | 相关性: 0.228]
等待中... [峰值: 0.001 | 相关性: 0.239]
等待中... [峰值: 0.001 | 相关性: 0.247]
等待中... [峰值: 0.001 | 相关性: 0.224]
等待中... [峰值: 0.001 | 相关性: 0.244]
等待中... [峰值: 0.001 | 相关性: 0.261]
等待中... [峰值: 0.001 | 相关性: 0.256]
等待中... [峰值: 0.001 | 相关性: 0.260]
等待中... [峰值: 0.001 | 相关性: 0.210]
等待中... [峰值: 0.001 | 相关性: 0.229]
等待中... [峰值: 0.001 | 相关性: 0.270]
等待中... [峰值: 0.001 | 相关性: 0.255]
等待中... [峰值: 0.001 | 相关性: 0.241]
等待中... [峰值: 0.001 | 相关性: 0.222]
等待中... [峰值: 0.001 | 相关性: 0.240]
等待中... [峰值: 0.001 | 相关性: 0.225]
等待中... [峰值: 0.001 | 相关性: 0.229]
等待中... [峰值: 0.001 | 相关性: 0.232]
等待中... [峰值: 0.001 | 相关性: 0.248]
等待中... [峰值:

[ERROR] [usrp2_iface.cpp:278] [USRP2] Control packet attempt 0, sequence number 460:
RuntimeError: no control response, possible packet loss
[ERROR] [USRP2] Control packet attempt 0, sequence number 460:
RuntimeError: no control response, possible packet loss
[ERROR] [usrp2_iface.cpp:278] [USRP2] Control packet attempt 1, sequence number 461:
RuntimeError: no control response, possible packet loss
[ERROR] [USRP2] Control packet attempt 1, sequence number 461:
RuntimeError: no control response, possible packet loss
[ERROR] [usrp2_iface.cpp:278] [USRP2] Control packet attempt 0, sequence number 464:
RuntimeError: no control response, possible packet loss
[ERROR] [USRP2] Control packet attempt 0, sequence number 464:
RuntimeError: no control response, possible packet loss
[ERROR] [usrp2_iface.cpp:278] [USRP2] Control packet attempt 0, sequence number 466:
RuntimeError: no control response, possible packet loss
[ERROR] [USRP2] Control packet attempt 0, sequence number 466:
RuntimeError: no

: 